In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
train_dir = "/content/drive/MyDrive/train"

In [2]:
test_dir = "/content/drive/MyDrive/test"

In [3]:
!pip install keras
!pip install tensorflow
!pip install transformers
import numpy as np
from transformers import AutoImageProcessor

In [4]:
import numpy as np

In [5]:
from PIL import Image

In [6]:
X_train = []
Y_train = []

In [7]:
import os

In [8]:
processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k")
for i in os.listdir(train_dir):
  for j in os.listdir(train_dir + "/" + i):
    img = Image.open(train_dir + "/" + i + "/" + j)
    inputs = processor(images=img, return_tensors="np")
    x = inputs["pixel_values"]
    x = x.flatten()
    X_train.append(x)
    Y_train.append(i)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


In [9]:
X_test = []
Y_test = []
for i in os.listdir(test_dir):
  for j in os.listdir(test_dir + "/" + i):
    img = Image.open(test_dir + "/" + i + "/" + j)
    inputs = processor(images=img, return_tensors="np")
    x = inputs["pixel_values"]
    x = x.flatten()
    X_test.append(x)
    Y_test.append(i)

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import numpy as np

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, Y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(Y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.7914893617021277


In [11]:
import joblib
joblib.dump(model, "model.pkl")

['model.pkl']

In [12]:
import gradio as gr
import numpy as np
from PIL import Image
from transformers import AutoImageProcessor
import joblib

processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")
clf = joblib.load("model.pkl")   # your scikit-learn model

def classify_image(img: Image.Image):
    proc = processor(images=img, return_tensors="np")  # returns 'pixel_values' as numpy
    x = proc["pixel_values"][0]                        # shape: (3, 224, 224)
    x = x.flatten()                                    # shape: (150528,)
    pred = clf.predict([x])[0]
    return str(pred)

interface = gr.Interface(
    fn=classify_image,
    inputs=gr.Image(type="pil"),
    outputs="text",
    title="Image Classification (joblib pipeline)",
    description="Upload an image and classify it using a custom model trained on ViT-processed features."
)
interface.launch(debug=True)


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9801e86e14db446e29.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://9801e86e14db446e29.gradio.live
